In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler

In [12]:
# Surpress warnings:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

In [18]:
data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/Ames_Housing_Sales.csv")
data.head()
print(data.shape)
data.dtypes.value_counts()

(1379, 80)


str        43
float64    21
int64      16
Name: count, dtype: int64

## Giải thích ý nghĩa của đoạn code
Đoạn code này **chưa thực hiện One-Hot Encoding**, mà chỉ dùng để **ước lượng số cột mới sẽ tăng thêm** nếu encode các biến phân loại.
### Các bước chính
1. **Lấy các cột phân loại**
   - `categorical_cols = data.select_dtypes(include=['object']).columns`
   - Mục tiêu: xác định các cột kiểu chuỗi (categorical).

2. **Đếm số giá trị duy nhất của từng cột phân loại**
   - `num_ohc_cols = data[categorical_cols].apply(lambda x: x.nunique()).sort_values(ascending=False)`
   - Mục tiêu: biết mỗi cột có bao nhiêu nhóm (category).

3. **Loại các cột không cần encode**
   - `small_num_ohc_cols = num_ohc_cols.loc[num_ohc_cols > 1]`
   - Nếu cột chỉ có 1 giá trị duy nhất thì không tạo thêm thông tin khi encode.

4. **Ước lượng số cột tăng thêm sau OHE**
   - `small_num_ohc_cols -= 1`
   - Với mỗi cột có $k$ nhóm, số cột dummy thường là $k-1$ (tránh đa cộng tuyến).

5. **Tính tổng số cột mới dự kiến**
   - `small_num_ohc_cols.sum()`
   - Đây là tổng số cột có thể tăng thêm sau khi One-Hot Encoding.

### Ý nghĩa thực tế
- Giúp bạn **đánh giá trước độ phình số chiều dữ liệu** (feature explosion).
- Hỗ trợ quyết định nên encode toàn bộ hay chọn lọc cột.
- Giúp chuẩn bị tài nguyên tính toán và tránh mô hình quá phức tạp.

### Lưu ý
- `small_num_ohc_cols.sum()` là **tổng số cột mới dự kiến**.
- Không phải số cột categorical ban đầu, cũng không phải dữ liệu đã encode.

In [ ]:
# Select the object (string) columns
categorical_cols = data.select_dtypes(include=['object']).columns
print("Số lượng cột trong categorical_cols:", len(categorical_cols))

# Determine how many extra columns would be created
num_ohc_cols = (data[categorical_cols]
                .apply(lambda x: x.nunique())
                .sort_values(ascending=False))
# No need to encode if there is only one value
small_num_ohc_cols = num_ohc_cols.loc[num_ohc_cols>1]

# Number of one-hot columns is one less than the number of categories
small_num_ohc_cols -= 1

# This is 215 columns, assuming the original ones are dropped. 
# This is quite a few extra columns!
small_num_ohc_cols.sum()
print("Số lượng cột trong small_num_ohc_cols:", small_num_ohc_cols.sum())

Số lượng cột trong categorical_cols: 43
['Alley', 'BldgType', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'CentralAir', 'Condition1', 'Condition2', 'Electrical', 'ExterCond', 'ExterQual', 'Exterior1st', 'Exterior2nd', 'Fence', 'FireplaceQu', 'Foundation', 'Functional', 'GarageCond', 'GarageFinish', 'GarageQual', 'GarageType', 'Heating', 'HeatingQC', 'HouseStyle', 'KitchenQual', 'LandContour', 'LandSlope', 'LotConfig', 'LotShape', 'MSZoning', 'MasVnrType', 'MiscFeature', 'Neighborhood', 'PavedDrive', 'PoolQC', 'RoofMatl', 'RoofStyle', 'SaleCondition', 'SaleType', 'Street', 'Utilities']
Số lượng cột trong num_ohc_cols: 43
Số lượng cột trong small_num_ohc_cols: 204


## Giải thích đoạn code One-Hot Encoding
Đoạn code này thực hiện **mã hóa One-Hot** cho các cột phân loại và ghép lại vào dataframe mới `data_ohc`.

### Luồng xử lý
1. **Sao chép dữ liệu gốc**
   - `data_ohc = data.copy()`
   - Mục đích: tránh thay đổi trực tiếp `data`.

2. **Khởi tạo encoder**
   - `LabelEncoder()` dùng để đổi nhãn chuỗi thành số nguyên.
   - `OneHotEncoder()` dùng để tạo vector one-hot từ các số nguyên đó.

3. **Lặp qua từng cột phân loại**
   - `for col in num_ohc_cols.index:`
   - Mỗi vòng lặp xử lý một cột categorical.

4. **Mã hóa và thay thế cột**
   - `le.fit_transform(...)`: chuyển giá trị chuỗi của cột thành số.

   Ví dụ cột `Color`:
   ```python
   ["Red", "Blue", "Green", "Blue", "Red"]
   ```
   Mapping (minh họa):
   - `Red -> 0`
   - `Blue -> 1`
   - `Green -> 2`

   - `drop(col, axis=1)`: xóa cột gốc khỏi `data_ohc`.
   - `ohc.fit_transform(...)`: tạo ma trận one-hot (dạng sparse).

   One-hot (minh họa):
   - `Red -> [1, 0, 0]`
   - `Blue -> [0, 1, 0]`
   - `Green -> [0, 0, 1]`

5. **Tạo tên cột mới và ghép vào dataframe**
   - Tạo tên cột dạng `tenCot_0`, `tenCot_1`, ...
   - Chuyển ma trận one-hot sang `DataFrame`.
   - `pd.concat(..., axis=1)`: nối các cột mới vào `data_ohc`.

### Kết quả
- `data_ohc` là dataset đã thay các cột categorical bằng các cột one-hot.
- Số cột thường tăng lên đáng kể sau bước này.

### Lưu ý
- Nếu bạn muốn bỏ bớt 1 cột mỗi nhóm để tránh đa cộng tuyến, cấu hình `OneHotEncoder(drop='first')`.
- Có thể thay toàn bộ quy trình này bằng `pd.get_dummies(...)` để viết gọn hơn.

In [29]:
# Copy of the data
data_ohc = data.copy()

# The encoders
le = LabelEncoder()
ohc = OneHotEncoder()

for col in num_ohc_cols.index:
    
    # Integer encode the string categories
    dat = le.fit_transform(data_ohc[col]).astype(int)
    
    # Remove the original column from the dataframe
    data_ohc = data_ohc.drop(col, axis=1)

    # One hot encode the data--this returns a sparse array
    new_dat = ohc.fit_transform(dat.reshape(-1,1))

    # Create unique column names
    n_cols = new_dat.shape[1]
    col_names = ['_'.join([col, str(x)]) for x in range(n_cols)]

    # Create the new dataframe
    new_df = pd.DataFrame(new_dat.toarray(), 
                          index=data_ohc.index, 
                          columns=col_names)

    # Append the new data to the data. frame
    data_ohc = pd.concat([data_ohc, new_df], axis=1)

In [ ]:
data_ohc.shape[1] - data.shape[1]

# Remove the string columns from the dataframe
data = data.drop(num_ohc_cols.index, axis=1)

80
37


In [33]:
data.head()
data_ohc.head()

,1stFlrSF,2ndFlrSF,3SsnPorch,BedroomAbvGr,BsmtFinSF1,BsmtFinSF2,BsmtFullBath,BsmtHalfBath,BsmtUnfSF,EnclosedPorch,...,LandSlope_2,CentralAir_0,CentralAir_1,Alley_0,Alley_1,Alley_2,Street_0,Street_1,Utilities_0,Utilities_1
0,856.0,854.0,0.0,3,706.0,0.0,1,0,150.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
1,1262.0,0.0,0.0,3,978.0,0.0,0,1,284.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
2,920.0,866.0,0.0,3,486.0,0.0,1,0,434.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
3,961.0,756.0,0.0,3,216.0,0.0,1,0,540.0,272.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
4,1145.0,1053.0,0.0,4,655.0,0.0,1,0,490.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


## Giải thích: Chia Train/Test cho 2 phiên bản dữ liệu

Đoạn code này **tạo 2 bộ train/test riêng biệt** để so sánh hiệu suất mô hình:
- **Một bộ** từ dữ liệu gốc (chưa one-hot encode)
- **Một bộ** từ dữ liệu đã one-hot encode

### Các bước chi tiết

#### 1. Xác định cột target
```python
y_col = 'SalePrice'
```
- Cột mục tiêu dùng để dự đoán là `SalePrice` (giá bán).

#### 2. Chia dữ liệu gốc (chưa encode)
```python
feature_cols = [x for x in data.columns if x != y_col]
X_data = data[feature_cols]           # Features (tất cả cột trừ target)
y_data = data[y_col]                  # Target (cột SalePrice)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, 
                                                    test_size=0.3, random_state=42)
```
- Tách features và target khỏi dữ liệu gốc.
- Chia 70% train, 30% test.
- `random_state=42` đảm bảo chia data giống nhau mỗi lần chạy (tái tạo được).

#### 3. Chia dữ liệu đã one-hot encode
```python
feature_cols = [x for x in data_ohc.columns if x != y_col]
X_data_ohc = data_ohc[feature_cols]
y_data_ohc = data_ohc[y_col]

X_train_ohc, X_test_ohc, y_train_ohc, y_test_ohc = train_test_split(X_data_ohc, y_data_ohc, 
                                                    test_size=0.3, random_state=42)
```
- Tương tự nhưng dùng `data_ohc` (đã one-hot encode).
- Tạo bộ train/test riêng: `X_train_ohc`, `X_test_ohc`, `y_train_ohc`, `y_test_ohc`.

### Kết quả

| Biến | Mô tả |
|------|-------|
| `X_train, y_train` | Dữ liệu train gốc (70%) |
| `X_test, y_test` | Dữ liệu test gốc (30%) |
| `X_train_ohc, y_train_ohc` | Dữ liệu train one-hot (70%) |
| `X_test_ohc, y_test_ohc` | Dữ liệu test one-hot (30%) |

### Ý nghĩa

- **So sánh mô hình**: Có thể huấn luyện cùng loại mô hình trên 2 phiên bản dữ liệu khác nhau.
- **Đánh giá tác dụng encoding**: Kiểm tra xem one-hot encoding có cải thiện độ chính xác hay không.
- **Cùng seed**: Dùng `random_state=42` để chia data giống nhau → so sánh công bằng.

In [34]:
y_col = 'SalePrice'

# Split the data that is not one-hot encoded
feature_cols = [x for x in data.columns if x != y_col]
X_data = data[feature_cols]
y_data = data[y_col]

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, 
                                                    test_size=0.3, random_state=42)
# Split the data that is one-hot encoded
feature_cols = [x for x in data_ohc.columns if x != y_col]
X_data_ohc = data_ohc[feature_cols]
y_data_ohc = data_ohc[y_col]

X_train_ohc, X_test_ohc, y_train_ohc, y_test_ohc = train_test_split(X_data_ohc, y_data_ohc, 
                                                    test_size=0.3, random_state=42)

In [36]:
# Compare the indices to ensure they are identical
(X_train_ohc.index == X_train.index).all()
X_train

,1stFlrSF,2ndFlrSF,3SsnPorch,BedroomAbvGr,BsmtFinSF1,BsmtFinSF2,BsmtFullBath,BsmtHalfBath,BsmtUnfSF,EnclosedPorch,...,OverallCond,OverallQual,PoolArea,ScreenPorch,TotRmsAbvGrd,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,YrSold
461,630.0,0.0,0.0,1,515.0,0.0,1,0,115.0,0.0,...,8,4,0.0,0.0,3,630.0,0.0,1970,2002,2009
976,845.0,0.0,0.0,3,0.0,0.0,0,0,0.0,0.0,...,3,4,0.0,0.0,5,0.0,186.0,1957,1957,2009
1128,728.0,728.0,0.0,3,0.0,0.0,0,0,728.0,0.0,...,5,6,0.0,0.0,8,728.0,100.0,2005,2005,2008
904,561.0,668.0,0.0,2,285.0,0.0,0,0,276.0,0.0,...,6,6,0.0,0.0,5,561.0,150.0,1980,1980,2009
506,1601.0,0.0,0.0,3,1358.0,0.0,1,0,223.0,0.0,...,5,8,0.0,0.0,6,1581.0,180.0,2001,2002,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,855.0,601.0,0.0,3,311.0,0.0,0,0,544.0,0.0,...,5,6,0.0,0.0,7,855.0,26.0,1978,1978,2010
1130,815.0,875.0,0.0,3,0.0,0.0,0,0,815.0,330.0,...,6,7,0.0,0.0,7,815.0,0.0,1916,1950,2006
1294,1661.0,0.0,0.0,3,831.0,0.0,1,0,161.0,0.0,...,6,6,0.0,178.0,8,992.0,0.0,1955,1996,2008
860,742.0,742.0,0.0,3,0.0,0.0,0,0,742.0,0.0,...,5,6,0.0,0.0,8,742.0,36.0,2005,2005,2009


## So sánh hiệu suất: Dữ liệu gốc vs One-Hot Encoding

Đoạn code này **huấn luyện mô hình Linear Regression trên 2 phiên bản dữ liệu** và **so sánh độ lỗi** để đánh giá hiệu quả của One-Hot Encoding.

### Quy trình thực hiện

#### 1. Khởi tạo mô hình và lưu trữ kết quả
```python
LR = LinearRegression()
error_df = list()
```
- Tạo mô hình Linear Regression.
- Tạo list để lưu kết quả lỗi của từng trường hợp.

#### 2. Train trên dữ liệu gốc (chưa encode)
```python
LR = LR.fit(X_train, y_train)
y_train_pred = LR.predict(X_train)
y_test_pred = LR.predict(X_test)
```
- Huấn luyện mô hình trên `X_train` (dữ liệu gốc).
- Dự đoán trên cả train và test set.

**Tính lỗi MSE:**
```python
error_df.append(pd.Series({'train': mean_squared_error(y_train, y_train_pred),
                           'test' : mean_squared_error(y_test,  y_test_pred)},
                           name='no enc'))
```
- Tính Mean Squared Error (MSE) cho train và test.
- Lưu kết quả với tên `'no enc'` (không encode).

#### 3. Train trên dữ liệu one-hot encode
```python
LR = LR.fit(X_train_ohc, y_train_ohc)
y_train_ohc_pred = LR.predict(X_train_ohc)
y_test_ohc_pred = LR.predict(X_test_ohc)
```
- Huấn luyện mô hình trên `X_train_ohc` (đã one-hot encode).
- Dự đoán trên cả train và test set.

**Tính lỗi MSE:**
```python
error_df.append(pd.Series({'train': mean_squared_error(y_train_ohc, y_train_ohc_pred),
                           'test' : mean_squared_error(y_test_ohc,  y_test_ohc_pred)},
                          name='one-hot enc'))
```
- Tính MSE cho train và test.
- Lưu kết quả với tên `'one-hot enc'`.

#### 4. Tổng hợp kết quả
```python
error_df = pd.concat(error_df, axis=1)
```
- Ghép 2 Series thành DataFrame để dễ so sánh.

### Kết quả mong đợi

DataFrame hiển thị MSE của 2 trường hợp:

|       | no enc | one-hot enc |
|-------|--------|-------------|
| train | xxx    | yyy         |
| test  | zzz    | www         |

### Cách đánh giá

- **MSE thấp hơn** = mô hình tốt hơn (dự đoán chính xác hơn).
- So sánh cột `test`:
  - Nếu `one-hot enc` < `no enc` → One-hot encoding **cải thiện** mô hình ✅
  - Nếu `one-hot enc` > `no enc` → One-hot encoding **không hiệu quả** ❌
- Kiểm tra overfitting: nếu `train` rất thấp nhưng `test` cao → mô hình overfitting

In [42]:
LR = LinearRegression()

# Storage for error values
error_df = list()

# Data that have not been one-hot encoded
LR = LR.fit(X_train, y_train)
y_train_pred = LR.predict(X_train)
y_test_pred = LR.predict(X_test)

error_df.append(pd.Series({'train': mean_squared_error(y_train, y_train_pred),
                           'test' : mean_squared_error(y_test,  y_test_pred)},
                           name='no enc'))

# Data that have been one-hot encoded
LR = LR.fit(X_train_ohc, y_train_ohc)
y_train_ohc_pred = LR.predict(X_train_ohc)
y_test_ohc_pred = LR.predict(X_test_ohc)

error_df.append(pd.Series({'train': mean_squared_error(y_train_ohc, y_train_ohc_pred),
                           'test' : mean_squared_error(y_test_ohc,  y_test_ohc_pred)},
                          name='one-hot enc'))

# Assemble the results
error_df = pd.concat(error_df, axis=1)
error_df

,no enc,one-hot enc
train,1.131507e+09,3.177267e+08
test,1.372182e+09,8.065328e+09


## Đánh giá kết quả

### Kết quả MSE

|       | no enc | one-hot enc |
|-------|--------|-------------|
| **train** | 1.13×10⁹ | 3.18×10⁸ |
| **test**  | 1.37×10⁹ | 8.07×10⁹ |

### Phân tích chi tiết

#### 1. Hiệu suất trên tập Train
- **no enc**: MSE = 1,131,507,000
- **one-hot enc**: MSE = 317,726,700 ✅
- **Kết luận**: One-hot encoding cho kết quả tốt hơn ~72% trên train set.

#### 2. Hiệu suất trên tập Test (QUAN TRỌNG)
- **no enc**: MSE = 1,372,182,000 ✅
- **one-hot enc**: MSE = 8,065,328,000 ❌
- **Kết luận**: One-hot encoding cho kết quả **TỆ HƠN ~488%** trên test set!

### Nhận xét tổng quan

⚠️ **Dấu hiệu Overfitting nghiêm trọng với One-Hot Encoding:**

1. **Mô hình one-hot đã overfitting**:
   - Train MSE rất thấp (317M)
   - Test MSE cực kỳ cao (8,065M)
   - Chênh lệch train/test: ~25 lần → overfitting nghiêm trọng

2. **Mô hình no encode tốt hơn**:
   - Train và test MSE gần nhau hơn
   - Test MSE thấp hơn nhiều so với one-hot
   - Khả năng tổng quát hóa tốt hơn

### Nguyên nhân và Giải pháp

**Nguyên nhân overfitting:**
- One-hot encoding tạo quá nhiều features (~215 cột mới)
- Mô hình học "thuộc lòng" train set nhưng không tổng quát hóa được
- Đa cộng tuyến giữa các cột one-hot

**Giải pháp:**
1. ✅ **Sử dụng mô hình gốc** (không encode) - cho kết quả test tốt hơn
2. 🔧 Giảm số features: chỉ encode một số cột quan trọng
3. 🔧 Regularization: dùng Ridge hoặc Lasso Regression
4. 🔧 Feature selection: loại bỏ features không quan trọng
5. 🔧 Tăng dữ liệu training

### Kết luận cuối cùng

**Không nên dùng One-Hot Encoding cho dataset này với Linear Regression** vì:
- Tạo quá nhiều features
- Gây overfitting nghiêm trọng
- Test performance giảm ~488%

→ **Khuyến nghị: Dùng dữ liệu gốc hoặc cân nhắc phương pháp encoding khác** (Target Encoding, Frequency Encoding, hoặc chọn lọc cột encode).

In [ ]:
# Mute the setting wtih a copy warnings
pd.options.mode.chained_assignment = None

scalers = {'standard': StandardScaler(),
           'minmax': MinMaxScaler(),
           'maxabs': MaxAbsScaler()}

training_test_sets = {
    'not_encoded': (X_train, y_train, X_test, y_test),
    'one_hot_encoded': (X_train_ohc, y_train_ohc, X_test_ohc, y_test_ohc)}


# Get the list of float columns, and the float data
# so that we don't scale something we already scaled. 
# We're supposed to scale the original data each time
mask = X_train.dtypes == float
float_columns = X_train.columns[mask]

# initialize model
LR = LinearRegression()

# iterate over all possible combinations and get the errors
errors = {}
for encoding_label, (_X_train, _y_train, _X_test, _y_test) in training_test_sets.items():
    for scaler_label, scaler in scalers.items():
        trainingset = _X_train.copy()  # copy because we dont want to scale this more than once.
        testset = _X_test.copy()
        trainingset[float_columns] = scaler.fit_transform(trainingset[float_columns])
        testset[float_columns] = scaler.transform(testset[float_columns])
        LR.fit(trainingset, _y_train)
        predictions = LR.predict(testset)
        key = encoding_label + ' - ' + scaler_label + 'scaling'
        errors[key] = mean_squared_error(_y_test, predictions)

errors = pd.Series(errors)
print(errors.to_string())
print('-' * 80)
for key, error_val in errors.items():
    print(key, error_val)

## Thử nghiệm kết hợp Encoding + Feature Scaling

Đoạn code này **kết hợp 2 kỹ thuật tiền xử lý** để tìm ra cấu hình tối ưu:
- **Encoding**: no encoding vs one-hot encoding
- **Scaling**: 3 phương pháp chuẩn hóa khác nhau

### Quy trình thực hiện

#### 1. Tắt cảnh báo copy warning
```python
pd.options.mode.chained_assignment = None
```
- Tắt warning khi copy và modify DataFrame để code chạy sạch hơn.

#### 2. Chuẩn bị các phương pháp Scaling
```python
scalers = {
    'standard': StandardScaler(),
    'minmax': MinMaxScaler(),
    'maxabs': MaxAbsScaler()
}
```

**3 loại scaler:**

| Scaler | Công thức | Khi nào dùng |
|--------|-----------|--------------|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Dữ liệu phân phối chuẩn, có outliers ít |
| **MinMaxScaler** | $x' = \frac{x - min}{max - min}$ | Scale về [0,1], nhạy với outliers |
| **MaxAbsScaler** | $x' = \frac{x}{max(\|x\|)}$ | Scale về [-1,1], giữ sparse data |

#### 3. Chuẩn bị 2 bộ dữ liệu
```python
training_test_sets = {
    'not_encoded': (X_train, y_train, X_test, y_test),
    'one_hot_encoded': (X_train_ohc, y_train_ohc, X_test_ohc, y_test_ohc)
}
```
- Tạo dictionary chứa 2 phiên bản: gốc và one-hot.

#### 4. Xác định cột cần scale
```python
mask = X_train.dtypes == float
float_columns = X_train.columns[mask]
```
- Chỉ scale cột numeric (float).
- Các cột one-hot (0/1) đã ở scale phù hợp nên không cần scale thêm.

#### 5. Vòng lặp thử nghiệm tất cả kết hợp
```python
for encoding_label, (_X_train, _y_train, _X_test, _y_test) in training_test_sets.items():
    for scaler_label, scaler in scalers.items():
        trainingset = _X_train.copy()
        testset = _X_test.copy()
        trainingset[float_columns] = scaler.fit_transform(trainingset[float_columns])
        testset[float_columns] = scaler.transform(testset[float_columns])
        LR.fit(trainingset, _y_train)
        predictions = LR.predict(testset)
        key = encoding_label + ' - ' + scaler_label + 'scaling'
        errors[key] = mean_squared_error(_y_test, predictions)
```

**Các bước trong vòng lặp:**
1. Copy dữ liệu để tránh modify gốc
2. **fit_transform** trên train (học mean/std/min/max)
3. **transform** trên test (dùng tham số từ train)
4. Train Linear Regression
5. Dự đoán và tính MSE
6. Lưu kết quả với key mô tả

#### 6. Hiển thị kết quả
```python
errors = pd.Series(errors)
print(errors.to_string())
```
- Chuyển dict thành Series để dễ đọc.
- In tất cả kết quả.

### Các kết hợp được test

Tổng cộng **6 kết hợp** (2 encoding × 3 scalers):

1. `not_encoded - standard scaling`
2. `not_encoded - minmax scaling`
3. `not_encoded - maxabs scaling`
4. `one_hot_encoded - standard scaling`
5. `one_hot_encoded - minmax scaling`
6. `one_hot_encoded - maxabs scaling`

### Ý nghĩa

- **Tìm cấu hình tốt nhất**: Kết hợp nào cho MSE test thấp nhất?
- **So sánh tác động scaling**: Scaling có giúp cải thiện không?
- **Hiểu tương tác**: Encoding + Scaling tác động như thế nào đến mô hình?

### Lưu ý quan trọng

⚠️ **Luôn fit scaler trên train, transform trên test**:
- ✅ Đúng: `scaler.fit_transform(train)` → `scaler.transform(test)`
- ❌ Sai: `scaler.fit_transform(test)` → data leakage!

## Quy trình đúng: Pipeline Encoding + Scaling

### ⚠️ Lưu ý QUAN TRỌNG về tránh Data Leakage

**Sơ đồ quy trình:**

```
Raw Data
   │
   ▼
[1] Train/Test Split
   │
   ├─────────────────────────┬──────────────────────────┐
   │                         │                          │
   ▼                         ▼                          │
Train Set (70%)          Test Set (30%)               │
   │                         │                          │
   ├─► [2] Fit Encoder      │                          │
   │        (học từ train)   │                          │
   │        │                │                          │
   │        ▼                │                          │
   │    [3] Transform        │                          │
   │    (apply encoder)      │                          │
   │        │                │                          │
   │        ▼                │                          │
   │    [4] Fit Scaler       │                          │
   │    (học từ train)       │                          │
   │        │                │                          │
   │        ▼                │                          │
   │    [5] Transform        │                          │
   │    (apply scaler)       │                          │
   │        │                │                          │
   │        ▼                ▼                          │
   │    [6] Train Model   Dùng Encoder từ Train ──────┤
   │                        │                          │
   │                        ▼                          │
   │                    Transform                      │
   │                        │                          │
   │                        ▼                          │
   │                    Dùng Scaler từ Train ─────────┤
   │                        │                          │
   │                        ▼                          │
   └────────────────────► [7] Predict / Evaluate
                             (Test MSE)
```

### Các bước chi tiết

| Bước | Action | Dữ liệu | Mục đích |
|------|--------|---------|---------|
| **1** | Split 70/30 | Raw Data | Chia train/test |
| **2** | **Fit** Encoder | Train Set | Học encoding rules từ train |
| **3** | **Transform** Encoder | Train Set | Apply rules vào train |
| **4** | **Fit** Scaler | Train Set (numeric) | Học mean/std/min/max từ train |
| **5** | **Transform** Scaler | Train Set | Apply scaling vào train |
| **6** | Train Model | Scaled Train Set | Fit Linear Regression |
| **7** | Transform Test | Test Set | Apply encoder & scaler từ train |
| **8** | Predict | Scaled Test Set | Dự đoán và tính MSE |

### ✅ Đúng vs ❌ Sai

**✅ ĐÚNG: Fit trên Train, Transform trên Test**
```python
# Train
scaler.fit(X_train[float_cols])
X_train_scaled = scaler.transform(X_train[float_cols])

# Test
X_test_scaled = scaler.transform(X_test[float_cols])  # Dùng fit từ train!
```

**❌ SAI: Fit riêng trên Test (Data Leakage!)**
```python
# KHÔNG LÀM CÁI NÀY!
scaler.fit(X_test[float_cols])  # ❌ Lỏm thông tin từ test!
X_test_scaled = scaler.transform(X_test[float_cols])
```

### Tại sao điều này quan trọng?

| Vấn đề | Nguyên nhân | Hậu quả |
|--------|-----------|---------|
| **Data Leakage** | Dùng thông tin từ test để fit encoder/scaler | Kết quả overly optimistic, không tổng quát |
| **Inconsistency** | Encoder/scaler khác nhau cho train/test | Mô hình lẫn lộn, dự đoán sai |
| **Invalid Evaluation** | Test metrics không phản ánh thực tế | Đánh giá mô hình không chính xác |

### Quy tắc vàng

> **Luôn fit trên Training Set, Transform trên Test Set.**
> 
> Encoder/Scaler là những công cụ được huấn luyện trên train data. 
> Test data chỉ được "transform" bằng những công cụ đó, không được fit lại.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


sns.set_context('talk')
sns.set_style('ticks')
sns.set_palette('dark')

ax = plt.axes()
# we are going to use y_test, y_test_pred
ax.scatter(y_test, y_test_pred, alpha=.5)

ax.set(xlabel='Ground truth', 
       ylabel='Predictions',
       title='Ames, Iowa House Price Predictions vs Truth, using Linear Regression');